# 第二部分：ONNX 部署详解

ONNX 是工业界最常用的**跨框架推理标准**，其目标是把你的 PyTorch 模型变成一种通用格式，让你可以在 **Python / C++ / 服务器 / 边缘设备** 上部署，并且可以对接多种后端加速器（CPU、CUDA、TensorRT、OpenVINO、TVM 等）。

下面从 **原理 → 导出 → 验证 → 推理 → 优化 → 排错** 层层展开。

---

## 一、ONNX 基本概念

### 1.1 ONNX 是什么

**ONNX = Open Neural Network Exchange**

它不是某一个公司的框架，而是由微软、Facebook、亚马逊等联合推动的**开放标准**，定位是：

> 一种用于表示机器学习模型的开放格式，使得模型可以在不同工具之间进行交换。

核心特征：

- **中立的计算图格式**：使用 Protocol Buffers 序列化一个有向无环图（DAG）
- **跨框架、跨硬件**：从 PyTorch 导出后，可以在 ONNX Runtime、TensorRT、OpenVINO、TVM、ncnn 等多种后端运行
- **算子标准库**：定义了超过 200 个标准算子（如 Conv、Gemm、Relu），覆盖视觉、NLP 绝大多数模型

### 1.2 ONNX 文件结构解剖

一个 `.onnx` 文件本质上是一个 `ModelProto` 对象，序列化后二进制存储。用 Netron（https://netron.app）可视化可以看到完整的图结构。其逻辑组成如下：

```text
model.onnx
├── ir_version           # IR 版本（如 7）
├── opset_import          # 使用的算子集及版本（如 ai.onnx v17）
├── producer_name         # 导出工具名（如 "pytorch"）
├── graph (GraphProto)    # 核心：计算图
│   ├── node (NodeProto)  # 计算节点，每个节点调用一个算子
│   │   ├── op_type       # 算子类型，如 "Conv", "Relu"
│   │   ├── input[]       # 输入 tensor 名称（可来自 initializer 或前驱节点）
│   │   ├── output[]      # 输出 tensor 名称
│   │   └── attribute[]   # 属性，如 kernel_shape, strides
│   ├── initializer       # 权重/偏置等参数（TensorProto）
│   ├── input             # 模型的输入声明（ValueInfoProto）
│   ├── output            # 模型的输出声明
│   └── value_info        # 中间张量的形状、数据类型信息（可选）
└── metadata_props        # 额外的元数据
```

理解这张图的关键：

- **node** 是计算操作，它通过名称引用 **initializer**（常量权重）和其他 **node 的输出**
- **initializer** 就是已经训练好的权重张量
- **input/output** 定义了模型对外暴露的接口

### 1.3 ONNX 算子与 Opset

ONNX 的算子集合会不断演进，每个版本称为一个 **opset**（例如 opset 13、17）。不同 opset 支持的算子数量和定义可能不同。PyTorch 导出时需要指定 `opset_version`，如果不指定，默认使用较旧的版本（如 opset 9），可能导致一些新算子不被识别而导出失败。

**建议**：使用较新且被目标后端支持的 opset，通常 13~17 是常用区间。若遇到算子不支持，提高 opset 是第一步排查手段。

### 1.4 ONNX 的优势总结

| 优势 | 具体含义 |
|------|----------|
| **跨平台** | Windows、Linux、macOS、Android、iOS 均有运行时 |
| **跨框架** | PyTorch、TensorFlow、Keras、MXNet、scikit-learn 等均可导出 |
| **后端丰富** | ONNX Runtime、TensorRT、OpenVINO、TVM、ncnn 均可直接加载 |
| **图优化** | ONNX Runtime 内置多种图优化 Pass，可离线或在线加速 |
| **工程生态** | Triton Server、FastDeploy 等直接集成 ONNX 模型 |
| **可读可调** | 用 Netron 可直接查看图，方便调试和验证 |

---

## 二、PyTorch 导出 ONNX

### 2.1 基本导出流程

导出前需要准备好：

1. **已训练的模型**，并切换到 `eval()` 模式
2. **一个示例输入张量**，shape 和 dtype 必须与实际推理输入一致

```python
import torch

model = YourModel()
model.load_state_dict(torch.load("weights.pth"))
model.eval()   # ⚠️ 必须设置，否则 BN/Dropout 行为异常

dummy_input = torch.randn(1, 3, 224, 224)   # 示例输入

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    opset_version=17,            # 算子集版本
    input_names=["input"],       # 输入节点名
    output_names=["output"],     # 输出节点名
    do_constant_folding=True,    # 常量折叠优化
    verbose=False
)
```

**参数详解**：

- `opset_version`：控制导出到 ONNX 的算子版本。版本越高，支持的新算子越多，但需确认后端（如 TensorRT）是否支持
- `input_names / output_names`：为模型的输入/输出张量命名，后续推理时通过名称喂数据
- `do_constant_folding`：导出时自动计算图中的常量表达式，简化图并减小体积（推荐开启）
- `verbose`：若为 `True`，会打印导出过程中的图 IR，有助于 debug

### 2.2 动态 Batch / 动态 Shape 导出

生产中的 batch size 很少固定，我们需要让导出的 ONNX 模型接受任意 batch。这通过 `dynamic_axes` 实现。

```python
torch.onnx.export(
    model,
    dummy_input,
    "model_dynamic.onnx",
    opset_version=13,   # 动态维度至少需要 opset 11
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},   # 将第 0 维标记为动态
        "output": {0: "batch_size"}
    }
)
```

`dynamic_axes` 字典的格式为 `{张量名: {维度索引: 轴名称}}`。可以同时标记多个维度为动态，例如支持可变长序列（NLP）、可变分辨率（图像）：

```python
dynamic_axes={
    "input": {0: "batch", 2: "height", 3: "width"},
    "output": {0: "batch"}
}
```

**注意**：动态维度只是告诉 ONNX “该维度可以变化”，真实的推理时仍受限于内存和硬件。同时，某些后端（如 TensorRT）构建引擎时需要预定义 min/opt/max shape 范围，并非完全自由。

### 2.3 控制流模型的导出

如果你的模型 `forward` 中包含 `if`、`for`、`while` 等控制流，直接 `torch.onnx.export` 可能会失败，因为导出器只能处理纯张量计算流。

**推荐方案**：先用 `torch.jit.script` 将模型转换为 TorchScript，再导出 ONNX。

```python
script_model = torch.jit.script(model)
torch.onnx.export(script_model, dummy_input, "model.onnx", ...)
```

TorchScript 已将控制流（如 `if`）转换为子图结构，ONNX 导出时可以将其翻译为 `If` 和 `Loop` 算子。但如果模型包含非常复杂的 Python 逻辑（如动态创建类、调用外部库），则几乎无法导出，需要重构模型代码。

### 2.4 导出时的常见陷阱

- **忘记 `model.eval()`**：导致 Dropout 仍随机置零，BatchNorm 仍在计算当前 batch 的统计量，推理结果既不稳定也变慢。
- **输入 dtype 不匹配**：导出时 `dummy_input` 是 float32，但模型期望 float16 或其他类型，可能导致导出的模型在 ONNX Runtime 推理时 shape 或数值异常。
- **多输出处理**：若模型返回多个张量（例如 `return feat1, feat2`），只需在 `output_names` 中声明对应数量的名字，ONNX 会自动捕获所有输出。
- **包含 in-place 操作**：PyTorch 的 in-place 操作（如 `x.relu_()`）在 ONNX 中大多有对应支持，但有些复杂情况可能导致图不正确，建议使用非 in-place 版本。

---

## 三、ONNX 模型验证

导出后必须验证模型的正确性，否则后续出问题极难定位。

### 3.1 使用 onnx.checker 进行基础检查

```python
import onnx

model_onnx = onnx.load("model.onnx")
onnx.checker.check_model(model_onnx)
print("ONNX model is valid!")
```

`check_model` 会检查图结构完整性、节点输入/输出类型一致性、算子存在性等。如果报错，会明确提示出错节点。

### 3.2 形状推断验证

为了确认 ONNX 推理时各中间张量的形状是否正确，可以运行形状推断：

```python
from onnx import shape_inference
inferred_model = shape_inference.infer_shapes(model_onnx)
onnx.save(inferred_model, "model_inferred.onnx")
```

这样用 Netron 查看就能看到每个节点的输出 shape，有助于排查动态维度设置是否正确。

### 3.3 数值精度对比验证

这是最关键的一步：**比较原 PyTorch 模型和 ONNX 模型的输出数值是否一致**。

```python
import numpy as np
import onnxruntime as ort

# 准备相同输入
dummy_input_np = dummy_input.cpu().numpy()

# PyTorch 输出
with torch.no_grad():
    torch_output = model(dummy_input).cpu().numpy()

# ONNX Runtime 输出
session = ort.InferenceSession("model.onnx")
onnx_output = session.run(None, {"input": dummy_input_np})[0]

# 比较
np.testing.assert_allclose(torch_output, onnx_output, rtol=1e-3, atol=1e-5)
print("Outputs are close!")
```

如果出现较大误差（ > 1e-3 相对误差），常见原因：
- 模型中包含了不稳定的自定义算子，或导出 opset 版本导致算子实现不同
- `BatchNorm` 的 `momentum` 等参数计算路径在 ONNX 和 PyTorch 有细微差异（通常很小）
- 使用了 FP16 而实际导出是 FP32，但 PyTorch 端可能是混合精度，检查类型

---

## 四、ONNX Runtime 推理

ONNX Runtime (ORT) 是微软开发的高性能推理引擎，支持多种 Execution Provider，是 ONNX 模型最常用的运行时。

### 4.1 安装

```bash
pip install onnxruntime-gpu   # GPU 推理
# 或
pip install onnxruntime       # 仅 CPU
```

若需同时支持 CPU 和 GPU，可安装 `onnxruntime-gpu`，它包含 CPU 能力。

### 4.2 基本 Python 推理

```python
import onnxruntime as ort
import numpy as np

# 创建推理会话
session = ort.InferenceSession("model.onnx")

# 查看输入输出信息
for inp in session.get_inputs():
    print(f"Input: {inp.name}, shape: {inp.shape}, dtype: {inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name}, shape: {out.shape}, dtype: {out.type}")

# 准备输入数据（必须与导出时的 dtype 一致）
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)

# 执行推理
outputs = session.run(None, {"input": input_data})
# outputs 是一个列表，顺序与 output_names 一致
print("Output shape:", outputs[0].shape)
```

`session.run` 的第一个参数若为 `None`，则返回所有输出；若指定名称列表（如 `["output"]`），则只返回对应输出。

### 4.3 指定 Execution Provider

```python
# 仅 CPU
session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])

# GPU
session = ort.InferenceSession("model.onnx", providers=["CUDAExecutionProvider"])

# GPU 带 fallback
session = ort.InferenceSession("model.onnx", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
```

还可以设置 provider 选项，例如指定 GPU 设备 ID：

```python
providers = [("CUDAExecutionProvider", {"device_id": 1})]
session = ort.InferenceSession("model.onnx", providers=providers)
```

### 4.4 C++ 推理（简述）

在生产环境中，常需要在 C++ 服务中调用 ONNX Runtime，以获得更低延迟和更好的资源控制。基本步骤与 Python 类似：

```cpp
#include <onnxruntime_cxx_api.h>

Ort::Env env(ORT_LOGGING_LEVEL_WARNING, "test");
Ort::SessionOptions session_options;
session_options.SetIntraOpNumThreads(4);
Ort::Session session(env, "model.onnx", session_options);

// 获取输入信息，准备数据
Ort::AllocatorWithDefaultOptions allocator;
auto input_name = session.GetInputNameAllocated(0, allocator);
// 创建输入 tensor（需使用 Ort::MemoryInfo 和 Ort::Value）

// 执行推理
auto output_tensors = session.Run(Ort::RunOptions{nullptr}, input_names.data(), &input_tensor, 1, output_names.data(), 1);
```

详细可参考 ONNX Runtime 官方 C++ 示例。

---

## 五、性能优化

### 5.1 图优化等级

ONNX Runtime 内置多种图优化 Pass，可通过 `SessionOptions` 控制优化级别：

```python
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session = ort.InferenceSession("model.onnx", sess_options=so)
```

| 优化等级 | 说明 |
|----------|------|
| ORT_DISABLE_ALL | 不做任何优化 |
| ORT_ENABLE_BASIC | 基础优化（常量折叠、冗余节点消除） |
| ORT_ENABLE_EXTENDED | 扩展优化（算子融合、内存规划） |
| ORT_ENABLE_ALL | 最高级别（包含扩展优化，可能引入硬件相关优化） |

一般选择 `ORT_ENABLE_ALL` 即可得到最佳性能。

### 5.2 多线程调优

对于 CPU 推理，合理配置并行线程数非常重要：

```python
so = ort.SessionOptions()
so.intra_op_num_threads = 8   # 单个算子内部的并行度（如矩阵乘法）
so.inter_op_num_threads = 2   # 算子间的并行度（流水线）
```

设置后，ONNX Runtime 会利用 MKL/OpenBLAS 等多线程加速。但也要注意，如果服务本身已经有线程池，过多线程会造成超量争抢，需根据实际负载调优。

### 5.3 启用 TensorRT 后端加速（ONNX Runtime 集成）

如果你的 GPU 环境安装了 TensorRT 和 `onnxruntime-gpu`，可以直接让 ONNX Runtime 将 ONNX 子图编译为 TensorRT 引擎，实现混合执行：

```python
import onnxruntime as ort

providers = [
    ("TensorrtExecutionProvider", {
        "device_id": 0,
        "trt_fp16_enable": True,
        "trt_max_workspace_size": 2147483648,
    }),
    "CUDAExecutionProvider"
]

session = ort.InferenceSession("model.onnx", providers=providers)
```

ORT 会自动识别图中哪些部分可以用 TensorRT 加速，其余部分回退到普通 CUDA 执行。这种方式无需手动构建 TensorRT 引擎，非常适合快速验证和迭代。

### 5.4 动态 Shape 与内存优化

- **动态 shape**：使用 `dynamic_axes` 导出的模型，在 ORT 中推理时可以直接输入不同大小的张量，ORT 会自动处理内存分配。
- **内存模式**：`so.enable_mem_pattern = True`（默认）允许 ORT 一次性规划所有中间张量的内存，减少碎片和分配次数。
- **图序列化优化**：可以将优化后的 ORT 会话保存为 `.ort` 格式，下次直接加载跳过图优化阶段：
  ```python
  # 创建优化后的会话
  session_options.optimized_model_filepath = "model.optimized.ort"
  # 下次加载时直接使用这个文件，可减少启动时间
  ```

---

## 六、常见问题与解决策略

### 6.1 算子不支持（Unsupported Operator）

**现象**：
导出时提示某个 `aten::xxx` 算子无法转换为 ONNX 算子，或者 ONNX Runtime 加载时报 `Unsupported Operator`。

**解决**：
1. 提升 `opset_version`，高版本 opset 通常包含更多算子映射。
2. 先用 `torch.jit.script` 转换模型，再导出 ONNX，因为 TorchScript 可能将某些复杂算子分解为更基础的操作。
3. 对于确实无法映射的自定义算子，可以编写 ONNX Custom Op 并注册到 ONNX Runtime，但这属于高级操作。

### 6.2 动态 Shape 不生效

**现象**：
导出的 ONNX 模型似乎能输入不同 batch，但推理时报 shape 不匹配错误，或者输出结果错误。

**原因**：
- `dynamic_axes` 设置错误，未正确标记要变的维度。
- 模型中使用了 `Tensor.size()` 或 `Tensor.view()` 等包含动态维度计算的操作，trace 导出时这些被固化为常量，导致后续 shape 改变时出错。

**解决**：
- 检查导出代码中 `dynamic_axes` 是否正确命名且维度索引无误。
- 避免在模型中使用 shape 相关的 Python 数值计算，尽量用 PyTorch 的 `torch.jit.script` 导出，并用 `torch.jit.export` 标记。
- 若必须用 trace，可在 trace 后手动修正图中尺寸相关节点（难度大）。

### 6.3 精度下降

**现象**：
ONNX 模型推理结果与 PyTorch 差距较大（相对误差 > 1%）。

**常见原因**：
- 导出时强制使用了 FP16 而实际 PyTorch 模型是 FP32，应确认导出和推理的精度一致。
- 某些层的默认参数在 PyTorch 和 ONNX 算子中不同，例如 `Conv` 的 `padding` 方式。
- 使用了没有精确映射的激活函数，例如 PyTorch 的 `F.gelu` 与 ONNX 的 `Gelu` 在近似实现上可能略有差异（通常误差极小，可忽略）。

**解决**：
- 先在 FP32 下对齐精度，确保基本一致。
- 如果使用 ONNX Runtime + TensorRT 的 FP16 模式，可开启 `trt_fp16_enable`，但注意模型输出可能略有变化，需在业务可接受范围内。

---

## 七、工业部署场景与生态

| 场景 | 推荐方案 | 备注 |
|------|----------|------|
| 通用服务器（x86） | ONNX Runtime CPU (MKL/OpenBLAS) | 性能接近 LibTorch，但更轻量 |
| NVIDIA GPU 在线服务 | ONNX Runtime + TensorRT 后端 或 直接转 TensorRT | 追求极致性能则直接构建 TensorRT Engine |
| Intel 平台 | OpenVINO（可导入 ONNX 模型） | 针对 Intel CPU/GPU/VPU 优化 |
| 移动端 | ONNX Runtime Mobile 或 转为 ncnn/TNN | 需要量化、模型裁剪等 |
| 边缘设备（Jetson） | TensorRT（从 ONNX 构建） | 利用 GPU 加速 |
| 大规模推理集群 | Triton Inference Server + ONNX 模型 | 支持动态批处理、多模型管理 |

---

## 八、ONNX 部署总结

1. **PyTorch → ONNX**：一次导出，多处部署。关键要点是 `model.eval()`、正确的 `opset_version` 和 `dynamic_axes`。
2. **验证三步走**：结构检查 (`onnx.checker`) → 形状推断 → 数值对比。
3. **ONNX Runtime**：最通用的推理引擎，支持 Python/C++，CPU/GPU，并提供图优化和多线程配置。
4. **性能优化**：调整图优化等级、线程数，必要时借助 TensorRT 后端实现极致加速。
5. **常见陷阱**：算子支持、动态 shape 配置、精度对齐，需逐一验证。
